# Lab 6b: DataFrames, SQL and UDFs

Run the cells one at a time, from the top. The same notebook runs in the development container and in Google Colab: the first cell detects where it is.

## Set up the environment

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/gdrive")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"],
                   check=True,
                   stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==4.1.1"], check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    DATA = "/content/gdrive/MyDrive/big-data-labs/datasets"
    OUTPUT = "/content/gdrive/MyDrive/big-data-labs/output/lab06b"
else:
    DATA = "../../datasets"
    OUTPUT = "../shared/output/lab06b"

print("Google Colab" if IN_COLAB else "Development container", "| data in", DATA)

## Start Spark

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.master("local[*]").appName("lab06b-dataframes")
         .config("spark.sql.warehouse.dir", f"{OUTPUT}/spark-warehouse")
         .getOrCreate())
sc = spark.sparkContext
print("Spark", spark.version, "| cores:", sc.defaultParallelism)

## Read the CSV file

`inferSchema` reads the file once more to guess the type of each column.

In [ ]:
df = spark.read.csv(f"{DATA}/sample_data.csv", inferSchema=True, header=True)
df.show(5)

In [ ]:
print("Rows:", df.count(), "| columns:", len(df.columns), df.columns)
df.printSchema()

## Select and describe

In [ ]:
df.select("age", "mobile").show(5)

In [ ]:
df.describe().show()

## Add columns

`withColumn` gives a new DataFrame: `df` does not change.

In [ ]:
from pyspark.sql import functions as F

df.withColumn("age_after_10_yrs", F.col("age") + 10).withColumn(
    "age_double", F.col("age").cast("double")).show(5)

## Filter

In [ ]:
df.filter(F.col("mobile") == "Vivo").select("age", "ratings", "mobile").show()

In [ ]:
df.filter((F.col("mobile") == "Vivo") & (F.col("experience") > 10)).show()

## Distinct values and groups

In [ ]:
df.select("mobile").distinct().show()

In [ ]:
df.groupBy("mobile").count().orderBy(F.desc("count"), "mobile").show()

In [ ]:
df.groupBy("mobile").mean().orderBy("mobile").show()

In [ ]:
df.groupBy("mobile").agg(
    F.sum("experience").alias("experience_sum"),
    F.max("ratings").alias("ratings_max"),
    F.min("age").alias("age_min"),
).orderBy("mobile").show()

## SQL on a DataFrame

A temporary view gives a name to the DataFrame, so SQL can use it. It lasts as long as the Spark session.

In [ ]:
df.createOrReplaceTempView("phones")
spark.sql("SHOW TABLES").show()

In [ ]:
spark.sql("SELECT * FROM phones WHERE ratings > 3").show(5)

In [ ]:
spark.sql("""
    SELECT mobile, round(avg(ratings), 2) AS avg_ratings, count(*) AS users
    FROM phones
    GROUP BY mobile
    ORDER BY avg_ratings DESC, mobile
""").show()

## User-defined functions

A Python UDF runs a Python function on each row. A pandas UDF receives a whole batch of rows as a pandas Series, which moves the data between the JVM and Python with Arrow.

In [ ]:
def price_range(brand):
    if brand in ["Samsung", "Apple"]:
        return "High Price"
    if brand == "MI":
        return "Mid Price"
    return "Low Price"


brand_udf = F.udf(price_range, "string")
df.withColumn("price_range", brand_udf(F.col("mobile"))).show(5)

In [ ]:
age_udf = F.udf(lambda age: "young" if age <= 30 else "senior", "string")
df.withColumn("age_group", age_udf(F.col("age"))).show(5)

In [ ]:
import pandas as pd


@F.pandas_udf("long")
def remaining_yrs(age: pd.Series) -> pd.Series:
    return 100 - age


@F.pandas_udf("double")
def product(rating: pd.Series, experience: pd.Series) -> pd.Series:
    return rating * experience


df.withColumn("yrs_left", remaining_yrs(F.col("age"))).withColumn(
    "product", product(F.col("ratings"), F.col("experience"))).show(5)

## Duplicates and columns

In [ ]:
unique = df.dropDuplicates()
print("Rows:", df.count(), "| without duplicates:", unique.count())

In [ ]:
unique.drop("mobile").show(5)

## Save the results

Spark writes a folder, with one file per partition. `coalesce(1)` leaves a single partition, so the CSV folder has a single data file.

In [ ]:
unique.coalesce(1).write.mode("overwrite").csv(f"{OUTPUT}/sample_csv", header=True)
unique.write.mode("overwrite").parquet(f"{OUTPUT}/sample_parquet")
for folder in ["sample_csv", "sample_parquet"]:
    print(folder, [name for name in sorted(os.listdir(f"{OUTPUT}/{folder}")) if not name.startswith(".")])

In [ ]:
spark.read.parquet(f"{OUTPUT}/sample_parquet").count()

## Stop Spark

In [ ]:
spark.stop()